In [26]:
#require 'tree_sitter'
require 'tree_sitter_language_pack'

false

In [27]:
yaml = TreeSitter::Language.load('yaml', '/usr/lib64/libtree-sitter-yaml.so')
json = TreeSitter::Language.load('json', '/usr/lib64/libtree-sitter-json.so')
bash = TreeSitter::Language.load('bash', '/usr/lib64/libtree-sitter-bash.so')
javascript = TreeSitter::Language.load('javascript', '/usr/lib64/libtree-sitter-javascript.so')
toml = TreeSitter::Language.load('toml', '/usr/lib64/libtree-sitter-toml.so')
python = TreeSitter::Language.load('python', '/usr/lib64/libtree-sitter-python.so')

#<TreeSitter::Language:0x00007fdc505753d0>

In [28]:
puts TreeSitter::Parser.instance_methods(false).sort
# Does :set_language appear? If not, you have the wrong gem loaded.

cancellation_flag
cancellation_flag=
included_ranges
included_ranges=
language
language=
logger
logger=
parse
parse_string
parse_string_encoding
print_dot_graphs
reset


In [54]:
require "tree_sitter_language_pack"

parser = TreeSitterLanguagePack.get_parser("yaml")
tree   = parser.parse(File.read("roles/common/tasks/main.yml"))
root   = tree.root_node

# Walk mappings, keep raw scalars + byte offsets; ERROR nodes become findings,
# not exceptions. Jinja-bearing scalars go through the jinja2 parser second.

p tree

#<TreeSitterLanguagePack::Tree:0x00007fdc50117310>


#<TreeSitterLanguagePack::Tree:0x00007fdc50117310>

In [67]:
require "tree_sitter_language_pack"          # VERIFY: gem's require name
require "set"

module AnsibleParse
  Finding      = Struct.new(:path, :row, :col, :kind, :detail, keyword_init: true)
  ScalarValue  = Struct.new(:text, :row, :col, :quoted, keyword_init: true)
  VaultValue   = Struct.new(:path, :row, :label, keyword_init: true)

  # Canonical keyword set belongs in resolve/ (Spike 3 fixes it from
  # ansible-lint's tasks.json schema); a sketch-length subset here:
  TASK_KEYWORDS = %w[
    name when register loop loop_control until retries delay
    become become_user become_method notify changed_when failed_when
    ignore_errors tags vars environment no_log run_once delegate_to
    throttle listen action local_action block rescue always
  ].to_set

  def self.each_node(node, &blk)
    blk.call(node)
    node.named_children.each { |c| each_node(c, &blk) }    # VERIFY: named_children
  end

  # ------------------------------------------------------------------
  # One YAML file: tree + findings + offset-carrying structure
  # ------------------------------------------------------------------
  class YamlFile
    attr_reader :path, :source, :root, :findings

    def self.read(path)
      source = File.read(path)
      parser = TreeSitterLanguagePack.get_parser("yaml")     # VERIFY: namespace
      new(path, source, parser.parse(source)&.root_node).tap(&:census_errors)
    end

    def initialize(path, source, root)
      @path, @source, @root = path, source, root
      @findings = []
    end

    # Multi-document files (--- a ... --- b) surface as a "stream" node.
    def documents
      return [] if root.nil?
      root.kind == "stream" ? root.named_children : [root]
    end

    # -- shared traversal vocabulary (task/play/role layers all speak this) --
    WRAPPERS = %w[document block_node flow_node block_sequence_item].freeze
    ATTACHED = %w[anchor tag alias].freeze

    def unwrap(node)
      while node && WRAPPERS.include?(node.kind)
        inner = node.named_children.reject { |k| ATTACHED.include?(k.kind) }
        break if inner.empty?
        node = inner.first
      end
      node
    end

    def text_of(node)     node && unwrap(node).text                             end
    def mapping_of(node)  m = unwrap(node); m if m.kind.end_with?("_mapping")   end
    def sequence_of(node) s = unwrap(node); s if s.kind.end_with?("_sequence")  end
    def pairs_of(map)     map.named_children.select { |p| p.kind == "block_mapping_pair" } end
    def key_of(pair)      unwrap(pair.child_by_field_name("key"))               end  # VERIFY: field names
    def value_of(pair)    pair.child_by_field_name("value")                    end
    def tag_of(pair)      value_of(pair)&.named_children&.find { |k| k.kind == "tag" }&.text end

    # Tree -> Ruby. Every leaf is a ScalarValue (raw text + position, no
    # type opinions). !vault values become VaultValue, never decrypted.
    def to_ruby(node)
      node = unwrap(node)
      case node.kind
      when "block_mapping", "flow_mapping"
        pairs_of(node).each_with_object({}) do |pair, h|
          key = text_of(key_of(pair))
          val = value_of(pair)
          if tag_of(pair) == "!vault"
            payload = @source[val.start_byte...val.end_byte]    # offsets earn their keep
            label   = payload[/\$ANSIBLE_VAULT;[^;]*;[^;]*(?:;(\S+))?/, 1]
            h[key] = VaultValue.new(path: @path, row: val.start_point.row, label: label)
          else
            h[key] = val ? to_ruby(val) : nil
          end
        end
      when "block_sequence", "flow_sequence"
        node.named_children.map { |item| to_ruby(item) }
      else
        ScalarValue.new(text: node.text, row: node.start_point.row,
                        col: node.start_point.column,
                        quoted: node.kind.include?("quote"))
      end
    end

    # ERROR nodes are findings, never exceptions.
    def census_errors
      if root.nil?
        @findings << Finding.new(path: @path, row: 0, col: 0, kind: :parse_failed,
                                 detail: "nil tree (timeout or size limit)")
        return
      end
      AnsibleParse.each_node(root) do |n|
        if n.error? || n.missing?                       # VERIFY: error? / missing?
          @findings << Finding.new(path: @path, row: n.start_point.row,
                                   col: n.start_point.column, kind: :syntax_error,
                                   detail: n.to_sexp)
        end
      end
    end
  end

  # ------------------------------------------------------------------
  # Tasks: a slot (role tasks/, play tasks:, handlers:) is a
  # block_sequence of block_mappings; block/rescue/always nest the
  # same shape. Callers scope this to task slots — playbook plays
  # never reach this path.
  # ------------------------------------------------------------------
  class Task
    attr_reader :file, :row, :action, :raw, :children, :jinja

    def self.from_node(file, item)
      m = file.mapping_of(item)
      return nil unless m
      raw    = file.to_ruby(m)
      action = file.pairs_of(m).find do |pair|
        !TASK_KEYWORDS.include?(file.text_of(file.key_of(pair)))
      end

      task = new(file: file, row: m.start_point.row,
                 action: action && file.text_of(file.key_of(action)), raw: raw)
      task.wire_children(m)
      task.enrich_jinja
      task
    end

    def initialize(file:, row:, action:, raw:)
      @file, @row, @action, @raw = file, row, action, raw
      @children, @jinja = [], []
    end

    # block/rescue/always groups hold nested task sequences.
    def wire_children(mapping)
      %w[block rescue always].each do |group_key|
        pair = file.pairs_of(mapping).find { |p| file.text_of(file.key_of(p)) == group_key }
        seq  = pair && file.sequence_of(file.value_of(pair))
        next unless seq
        @children.concat(seq.named_children.filter_map { |item| Task.from_node(file, item) })
      end
    end

    # import_/include_ target: either `- import_tasks: foo.yml`
    # or `- include_tasks: { file: foo.yml }`.
    def import_target
      arg = raw[action]
      return arg.text if arg.is_a?(ScalarValue)
      arg.is_a?(Hash) ? arg["file"]&.text : nil
    end

    # Jinja2 second stage: every scalar whose text carries {{ }} / {% %}
    # is re-parsed with the jinja2 grammar; YAML offsets stay the frame.
    def enrich_jinja
      each_scalar(raw) { |sc| @jinja.concat(JinjaStage.fragments(sc)) }
      children.each(&:enrich_jinja)
    end

    private

    def each_scalar(obj, &blk)
      case obj
      when Hash         then obj.each_value { |v| each_scalar(v, &blk) }
      when Array        then obj.each { |v| each_scalar(v, &blk) }
      when ScalarValue  then blk.call(obj)
      end
    end
  end

  module TaskFile
    module_function

    def tasks_from_sequence(file, seq)
      seq.named_children.filter_map { |item| Task.from_node(file, item) }
    end
  end

  # ------------------------------------------------------------------
  # A whole role: slot entry files + include_/import_ chasing
  # ------------------------------------------------------------------
  class Role
    attr_reader :name, :dir, :tasks, :handlers, :defaults, :vars, :meta, :findings

    def initialize(dir)
      @dir, @name = dir, File.basename(dir)
      @tasks, @handlers, @findings = [], [], []
    end

    def scan
      @tasks    = slot_tasks("tasks/main")
      @handlers = slot_tasks("handlers/main")
      @defaults = raw_doc("defaults/main")
      @vars     = raw_doc("vars/main")
      @meta     = raw_doc("meta/main")
      self
    end

    private

    def first_existing(base)
      %w[yml yaml].lazy.map { |ext| "#{base}.#{ext}" }.find { |p| File.exist?(p) }
    end

    # Entry file + every include_/import_ target, cycle-guarded by realpath.
    def slot_tasks(entry)
      entry = first_existing(entry)
      return [] unless entry
      visited = Set[File.realpath(entry)]
      queue   = [entry]
      tasks   = []
      while (path = queue.shift)
        file = YamlFile.read(path)
        @findings.concat(file.findings)
        current = []
        file.documents.each do |doc|
          seq = file.sequence_of(doc)
          current.concat(TaskFile.tasks_from_sequence(file, seq)) if seq
        end
        tasks.concat(current)
        current.each do |t|
          next unless t.action.to_s =~ /(^|\.)(include_tasks|import_tasks)\z/
          ref = t.import_target
          next unless ref
          child = File.expand_path(ref, File.dirname(path))
          real  = begin; File.realpath(child); rescue StandardError; nil; end
          queue << child if real && visited.add?(real)
        end
      end
      tasks
    end

    def raw_doc(entry)
      entry = first_existing(entry)
      return nil unless entry
      file = YamlFile.read(entry)
      @findings.concat(file.findings)
      file.documents.map { |doc| file.to_ruby(doc) }
    end
  end

  # ------------------------------------------------------------------
  # A playbook: list of plays; each play owns task slots + role refs
  # ------------------------------------------------------------------
  Play = Struct.new(:file, :row, :hosts, :keywords, :roles, :slots, keyword_init: true)

  class Playbook
    SLOTS = %w[pre_tasks tasks post_tasks handlers].freeze

    attr_reader :path, :plays, :findings

    def initialize(path)
      @path, @plays, @findings = path, [], []
    end

    def parse
      file = YamlFile.read(@path)
      @findings.concat(file.findings)
      file.documents.each do |doc|
        seq = file.sequence_of(doc)
        next unless seq                       # a playbook is a list of plays
        seq.named_children.each do |item|
          m = file.mapping_of(item)
          next unless m
          raw = file.to_ruby(m)
          next unless raw.key?("hosts")        # a play carries hosts; refine in Spike 3
          play = Play.new(file: file, row: m.start_point.row, hosts: raw["hosts"],
                         keywords: raw, roles: role_refs(raw), slots: {})
          SLOTS.each do |slot|
            pair     = file.pairs_of(m).find { |p| file.text_of(file.key_of(p)) == slot }
            slot_seq = pair && file.sequence_of(file.value_of(pair))
            play.slots[slot] = slot_seq ? TaskFile.tasks_from_sequence(file, slot_seq) : []
          end
          @plays << play
        end
      end
      self
    end

    private

    # A role ref is either a bare string or a mapping (role:/tags:/vars:/when:).
    def role_refs(raw)
      (raw["roles"] || []).map do |r|
        r.is_a?(ScalarValue) ? { "role" => r.text } : { "role" => r["role"]&.text, "raw" => r }
      end
    end
  end

  # ------------------------------------------------------------------
  # Jinja2 second stage
  # ------------------------------------------------------------------
  module JinjaStage
    DELIMS     = /\{\{.*?\}\}|\{%.*?%\}/m
    JINJA_NAME = "jinja2"                       # VERIFY: language name in the pack

    class << self
      def parser
        @parser ||= TreeSitterLanguagePack.get_parser(JINJA_NAME)
      end
    end

    module_function

    def fragments(scalar)
      out = []
      scalar.text.to_s.scan(DELIMS) do
        frag, off = Regexp.last_match[0], Regexp.last_match.begin(0)
        root = parser.parse(frag)&.root_node
        errs = 0
        AnsibleParse.each_node(root) { |n| errs += 1 if n.error? } if root
        out << { text: frag, row: scalar.row, col: scalar.col + off, errors: errs }
      end
      out
    end
  end
end

# -- Usage ---------------------------------------------------------
role = AnsibleParse::Role.new("roles/gpu").scan
#pp role.tasks.first&.action    # => "ansible.builtin.package" (raw text)
#pp role.findings                # every ERROR with row/col — nothing raised
#pp role.tasks.first&.jinja      # => [{ text: "{{ packages | join(' ') }}", ... }]

#site = AnsibleParse::Playbook.new("local.yml").parse
#site.plays.each { |play| puts play.slots["tasks"].length }
p role.vars

(irb):4: warning: already initialized constant #<Class:0x00007fdc51c32dc0>::AnsibleParse::Finding
(irb):4: warning: previous definition of Finding was here
(irb):5: warning: already initialized constant #<Class:0x00007fdc51c32dc0>::AnsibleParse::ScalarValue
(irb):5: warning: previous definition of ScalarValue was here
(irb):6: warning: already initialized constant #<Class:0x00007fdc51c32dc0>::AnsibleParse::VaultValue
(irb):6: warning: previous definition of VaultValue was here
(irb):10: warning: already initialized constant #<Class:0x00007fdc51c32dc0>::AnsibleParse::TASK_KEYWORDS
(irb):10: warning: previous definition of TASK_KEYWORDS was here
(irb):46: warning: already initialized constant #<Class:0x00007fdc51c32dc0>::AnsibleParse::YamlFile::WRAPPERS
(irb):46: warning: previous definition of WRAPPERS was here
(irb):47: warning: already initialized constant #<Class:0x00007fdc51c32dc0>::AnsibleParse::YamlFile::ATTACHED
(irb):47: warning: previous definition of ATTACHED was here
(irb):24

nil
